# Preprocessing

In [4]:
import pandas as pd
import numpy as np

data = pd.read_parquet('../data/processed/tracks_enriched.parquet')
orig_data= pd.read_parquet('../data/processed/orig_data.parquet')
target = 'popularity'

In [ ]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse_output=False, dtype=int)
encoded_columns = encoder.fit_transform(data[['track_genre']])

encoded_data = pd.DataFrame(encoded_columns, columns=encoder.get_feature_names_out(['track_genre']), index=data.index)
data_copy = data.copy()

data = pd.concat([data_copy.drop(columns=['track_genre']), encoded_data], axis=1)
data.reset_index(drop=True)

,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,...,track_genre_spanish,track_genre_study,track_genre_swedish,track_genre_synth-pop,track_genre_tango,track_genre_techno,track_genre_trance,track_genre_trip-hop,track_genre_turkish,track_genre_world-music
0,100,11.963644,False,0.714,0.472,2,-7.375,1,0.0864,0.01300,...,0,0,0,0,0,0,0,0,0,0
1,99,12.200748,False,0.621,0.782,2,-5.548,1,0.0440,0.01250,...,0,0,0,0,0,0,0,0,0,0
2,98,11.999282,False,0.835,0.679,7,-5.329,0,0.0364,0.58300,...,0,0,0,0,0,0,0,0,0,0
3,98,12.073906,True,0.561,0.965,7,-3.673,0,0.0343,0.00383,...,0,0,0,0,0,0,0,0,0,0
4,97,12.092725,True,0.911,0.712,1,-5.105,0,0.0817,0.09010,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66803,0,12.431350,False,0.632,0.961,2,-3.576,0,0.0407,0.26900,...,0,0,0,0,0,0,0,0,0,0
66804,0,12.716699,False,0.803,0.838,0,-2.450,1,0.0949,0.59100,...,0,0,0,0,0,0,0,0,0,0
66805,0,12.223524,False,0.841,0.853,2,-2.131,1,0.0518,0.42400,...,0,0,0,0,0,0,0,0,0,0
66806,0,12.657744,False,0.667,0.733,9,-5.703,0,0.0427,0.75100,...,0,0,0,0,0,0,0,0,0,0


# Train Linear Regression Model (Oridinary Least Squares)

In [14]:
from sklearn.model_selection import cross_val_score, train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer

X = data.copy()
y = data_copy[target]
X = X.drop(columns=target)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
# Cross-validation on the traning set

pipe_lin_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

score_r2 = cross_val_score(pipe_lin_reg, X_train, y_train, cv=10, scoring='r2')
score_mae = cross_val_score(pipe_lin_reg, X_train, y_train, cv=10, scoring='neg_mean_absolute_error')
score_rmse = cross_val_score(pipe_lin_reg, X_train, y_train, cv=10, scoring='neg_root_mean_squared_error')

print("Plain: R2=", score_r2.mean())
print("Plain: MAE=", -score_mae.mean())
print("Plain: RMSE=", -score_rmse.mean())

Plain: R2= 0.7563437518251661
Plain: MAE= 6.071193585216873
Plain: RMSE= 8.758270275664197


In [22]:
# Now we actually train the model
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

pipe_lin_reg.fit(X_train, y_train)

y_train_pred = pipe_lin_reg.predict(X_train)
y_test_pred = pipe_lin_reg.predict(X_test)

print("Train R2:", r2_score(y_train, y_train_pred))
print("Test R2:", r2_score(y_test, y_test_pred))

print("Train MAE:", mean_absolute_error(y_train, y_train_pred))
print("Test MAE:", mean_absolute_error(y_test, y_test_pred))

print("Train RMSE:", root_mean_squared_error(y_train, y_train_pred))
print("Test RMSE:", root_mean_squared_error(y_test, y_test_pred))


Train R2: 0.7578667983543387
Test R2: 0.7547148337193049
Train MAE: 6.053275612611114
Test MAE: 6.054954482255019
Train RMSE: 8.733013618809677
Test RMSE: 8.73861397128506


In [23]:
corrs = X_train.corrwith(y_train).abs().sort_values(ascending=False)
print(corrs.head(20))

artist_fame_loo               0.846434
track_genre_iranian           0.208094
instrumentalness              0.186427
track_genre_detroit-techno    0.148401
track_genre_pop               0.146035
track_genre_grindcore         0.143460
track_genre_kids              0.141104
track_genre_k-pop             0.129883
track_genre_honky-tonk        0.128826
track_genre_chicago-house     0.126697
track_genre_idm               0.125828
track_genre_pop-film          0.123351
danceability                  0.121533
track_genre_romance           0.118465
track_genre_tango             0.117370
track_genre_hip-hop           0.115520
track_genre_chill             0.104982
track_genre_dance             0.102284
track_genre_electro           0.101096
speechiness                   0.098724
dtype: float64


In [24]:
X_train_no_artist = X_train.drop(columns=['artist_fame_loo'])
X_test_no_artist = X_test.drop(columns=['artist_fame_loo'])

pipe_lin_reg.fit(X_train_no_artist, y_train)

y_train_pred = pipe_lin_reg.predict(X_train_no_artist)
y_test_pred = pipe_lin_reg.predict(X_test_no_artist)

from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

print("Train R2:", r2_score(y_train, y_train_pred))
print("Test R2:", r2_score(y_test, y_test_pred))

print("Train MAE:", mean_absolute_error(y_train, y_train_pred))
print("Test MAE:", mean_absolute_error(y_test, y_test_pred))

print("Train RMSE:", root_mean_squared_error(y_train, y_train_pred))
print("Test RMSE:", root_mean_squared_error(y_test, y_test_pred))

Train R2: 0.5389100925207786
Test R2: 0.5396440902097563
Train MAE: 8.89111095114024
Test MAE: 8.849358901212485
Train RMSE: 12.051185550659577
Test RMSE: 11.97164344756654
